# Proof of faster brute force search in smaller SAT instances

In [1]:
from setup import setup_tools
from utils import run_command

setup_tools()

Setting project root: /home/krishnendu/Research/fv-invariant-mining
Updated PATH to include: [PosixPath('/home/krishnendu/Research/fv-invariant-mining/.tools/cadical/build'), PosixPath('/home/krishnendu/Research/fv-invariant-mining/.tools/abc'), PosixPath('/home/krishnendu/Research/fv-invariant-mining/.tools/oss-cad-suite/bin'), PosixPath('/home/krishnendu/Research/fv-invariant-mining/.tools/aiger'), PosixPath('/home/krishnendu/Research/fv-invariant-mining/.tools/cadiback')]


In [2]:
from typing import Dict, List, Set

import numpy as np
from aig_grapher import AIG
from tqdm.notebook import tqdm
def exhaustive_search_for_output_one(aig: AIG, num_inputs: int | None=None):
    """
    Extremely optimized exhaustive search using numpy vectorization and pre-compiled circuit.
    
    Args:
        aig: The AIG circuit to simulate
        num_inputs: Number of input bits to use in exhaustive search (default: len(aig.input_ids))
    
    Returns:
        Tuple of (input_assignment, output_value) if found, else None
    """
    if num_inputs is None:
        num_inputs = len(aig.input_ids)
    
    total_combinations = 2 ** num_inputs
    print(f"Searching through {total_combinations} input combinations...")
    
    # Build topological order and input mapping
    input_ids = aig.input_ids[:num_inputs]
    output_id, output_inverted = aig.outputs[0]
    
    # Pre-compute node evaluation order (topological sort)
    visited: Set[int] = set()
    eval_order: List[int] = []
    
    def topo_sort(node_id: int):
        if node_id in visited or node_id == 0:
            return
        visited.add(node_id)
        node = aig.nodes[node_id]
        if node.typ.name == 'INT':
            for fanin_id, _ in node.fanins:
                topo_sort(fanin_id)
        eval_order.append(node_id)
    
    topo_sort(output_id)
    
    # Process in chunks for better performance
    chunk_size = min(100000, total_combinations)
    
    for chunk_start in tqdm(range(0, total_combinations, chunk_size)):
        chunk_end = min(chunk_start + chunk_size, total_combinations)
        
        # Create all input combinations for this chunk
        combos = np.arange(chunk_start, chunk_end, dtype=np.uint64)
        
        # Initialize node values
        node_values: Dict[int, np.ndarray] = {}
        node_values[0] = np.zeros(1)
        
        for input_idx, input_id in enumerate(input_ids):
            node_values[input_id] = (combos >> input_idx) & 1
        
        # Evaluate internal nodes in topological order
        for node_id in eval_order:
            if node_id not in node_values:
                node = aig.nodes[node_id]
                fanin1_id, fanin1_inv = node.fanins[0]
                fanin2_id, fanin2_inv = node.fanins[1]
                
                val1 = node_values[fanin1_id]
                val2 = node_values[fanin2_id]
                
                if fanin1_inv:
                    val1 = 1 - val1
                if fanin2_inv:
                    val2 = 1 - val2
                
                node_values[node_id] = val1 & val2
        
        # Check output
        output_val: np.ndarray = node_values[output_id]
        if output_inverted:
            output_val = 1 - output_val
        
        # Search for first 1
        matches = np.where(output_val == 1)[0]
        if len(matches) > 0:
            solution_idx = matches[0]
            combo = chunk_start + solution_idx
            
            # Reconstruct input assignment for solution
            input_assignment = {0: 0}
            for bit_idx, input_id in enumerate(input_ids):
                input_assignment[input_id] = int((combo >> bit_idx) & 1)
            
            print(f"Found solution at combination {combo}: {input_assignment}")
            return input_assignment, int(output_val[solution_idx])
        
        # if chunk_end % 100000 == 0 or chunk_end == total_combinations:
        #     print(f"  Checked {chunk_end}/{total_combinations}...")
    
    print("No solution found in exhaustive search")
    return None

In [3]:
path = "/home/krishnendu/Research/fv-invariant-mining/data/circuits/fmt/miter_mult_14bit.fmt"

In [4]:
exhaustive_search_for_output_one(AIG(path.replace('fmt', 'aag')))

Searching through 268435456 input combinations...


  0%|          | 0/2685 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [25]:
from tools_fns import cadical_check


cadical_check(path.replace('fmt', 'cnf'))

KeyboardInterrupt: 